# 📉 04_error_analysis_eda: Model Failure Analysis
This notebook is used after training. We analyze the **Residuals** (Actual - Predicted) to see where the model is failing and hypothesize new features to improve performance.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pathlib
import joblib
import os
import sys

# Force the working directory to be the project root
project_root = pathlib.Path(os.getcwd()).parent if "notebooks" in os.getcwd() else pathlib.Path(os.getcwd())
os.chdir(project_root)

# Add project root to sys.path to allow importing from src
sys.path.append(str(project_root))

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Load processed data
X = pd.read_parquet("data/processed/X_train.parquet")
y = pd.read_parquet("data/processed/y_train.parquet")
df = X.copy()
df['target'] = y['target']

# Load the latest model
# Now using the project root context, the path matches exactly what we used in train.py
model_path = pathlib.Path("models/artifacts/baseline_ridge.pkl")
if model_path.exists():
    model = joblib.load(model_path)
    print(f"Model loaded successfully from {model_path}")
else:
    model = None
    print(f"Model not found at {model_path.absolute()}. Please run train.py first.")

## 1. The Residual Analysis
We plot the errors (Actual minus Predicted). A perfectly balanced model has a mean error of zero. If the errors are skewed, the model is biased.

In [ ]:
if model is not None:
    # Simple split for validation
    from sklearn.model_selection import train_test_split
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Generate predictions
    preds = model.predict(X_val)
    
    # Calculate residuals
    residuals = y_val.values - preds
    
    plt.figure(figsize=(12, 5))
    sns.histplot(residuals, kde=True, color='purple')
    plt.axvline(0, color='red', linestyle='--')
    plt.title("Residuals Distribution (Actual - Predicted)")
    plt.xlabel("Error (Seconds)")
    plt.show()
else:
    print("Model not loaded. Skipping plot.")